# Compare 4 Bandit Algorithms in One Plot

**Goal:** See how different bandit algorithms explore and exploit differently.

**What you'll learn:** The trade-offs between ε-greedy, UCB1, Thompson Sampling, and Softmax.

**Scenario:** 5 commodity sector strategies with Gaussian rewards. Which algorithm learns fastest?

**Algorithms we'll compare:**
1. **ε-greedy (ε=0.1):** Explore randomly 10% of the time
2. **UCB1:** Optimistic exploration using confidence bounds
3. **Thompson Sampling:** Bayesian probability matching
4. **Softmax (τ=0.1):** Probabilistic selection based on estimated values

---

In [ ]:
learning_objectives(["**ε-greedy (ε=0.1):** Explore randomly 10% of the time", "**UCB1:** Optimistic exploration using confidence bounds", "**Thompson Sampling:** Bayesian probability matching", "**Softmax (τ=0.1):** Probabilistic selection based on estimated values"])

## Setup

In [ ]:
import sys; sys.path.insert(0, '../../..')
from resources.notebook_style import apply_course_theme, learning_objectives, section_divider, callout, key_takeaways
from resources.graphics.plot_theme import apply_plot_theme

In [ ]:
!pip install -q numpy matplotlib pandas

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

np.random.seed(42)

In [ ]:
apply_course_theme()
apply_plot_theme()

## Create 5-Arm Bandit: Commodity Sectors

Each arm represents a commodity sector with Gaussian-distributed returns.

In [ ]:
# Define 5 commodity sectors: (name, mean_return, std_return)
sectors = [
    ('Energy', 0.003, 0.025),       # Volatile
    ('Metals', 0.006, 0.020),       # Best performer
    ('Agriculture', 0.002, 0.022),  # Low return
    ('Precious', 0.004, 0.018),     # Medium
    ('Livestock', 0.001, 0.015),    # Worst
]

n_arms = len(sectors)
n_rounds = 1000

print("Commodity Sector Bandit:")
print("\n{:<15} {:<18} {:<15}".format('Sector', 'Avg Weekly Return', 'Volatility'))
print("="*50)
for name, mean, std in sectors:
    print(f"{name:<15} {100*mean:+.2f}%             {100*std:.2f}%")

print(f"\nBest sector: {sectors[1][0]} ({100*sectors[1][1]:.2f}% avg return)")
print(f"Rounds: {n_rounds}")

## Helper: Get Reward

Simulate a weekly return from a sector.

In [ ]:
def get_reward(sector_idx):
    """Sample a reward from the sector's Gaussian distribution."""
    name, mean, std = sectors[sector_idx]
    return np.random.normal(mean, std)

## Algorithm 1: ε-Greedy (ε=0.1)

With probability ε, explore randomly. Otherwise, exploit the best-known arm.

In [ ]:
# ε-greedy
np.random.seed(42)
epsilon = 0.1

Q = np.zeros(n_arms)  # Estimated value of each arm
N = np.zeros(n_arms)  # Number of times each arm was pulled
rewards_eg = []
regret_eg = []
arm_counts_eg = np.zeros(n_arms)

best_mean = max([s[1] for s in sectors])

for t in range(n_rounds):
    # Explore or exploit?
    if np.random.random() < epsilon:
        arm = np.random.randint(n_arms)  # Explore
    else:
        arm = np.argmax(Q)  # Exploit
    
    # Pull arm and observe reward
    reward = get_reward(arm)
    rewards_eg.append(reward)
    arm_counts_eg[arm] += 1
    
    # Update estimate
    N[arm] += 1
    Q[arm] += (reward - Q[arm]) / N[arm]
    
    # Track regret
    regret = best_mean - sectors[arm][1]
    regret_eg.append(regret)

cumulative_regret_eg = np.cumsum(regret_eg)
print(f"ε-greedy (ε={epsilon}): Total regret = {cumulative_regret_eg[-1]:.4f}")

## Algorithm 2: UCB1

Upper Confidence Bound: optimistically estimate each arm's value and pick the highest.

In [ ]:
# UCB1
np.random.seed(42)

Q = np.zeros(n_arms)
N = np.zeros(n_arms)
rewards_ucb = []
regret_ucb = []
arm_counts_ucb = np.zeros(n_arms)

for t in range(n_rounds):
    # Pull each arm once first
    if t < n_arms:
        arm = t
    else:
        # UCB selection
        ucb_values = Q + np.sqrt(2 * np.log(t + 1) / (N + 1e-5))
        arm = np.argmax(ucb_values)
    
    reward = get_reward(arm)
    rewards_ucb.append(reward)
    arm_counts_ucb[arm] += 1
    
    N[arm] += 1
    Q[arm] += (reward - Q[arm]) / N[arm]
    
    regret = best_mean - sectors[arm][1]
    regret_ucb.append(regret)

cumulative_regret_ucb = np.cumsum(regret_ucb)
print(f"UCB1: Total regret = {cumulative_regret_ucb[-1]:.4f}")

## Algorithm 3: Thompson Sampling

Bayesian approach: maintain a Gaussian posterior for each arm and sample from it.

In [ ]:
# Thompson Sampling (Gaussian)
np.random.seed(42)

# Prior: N(0, 0.01^2)
mu = np.zeros(n_arms)
sigma_sq = np.ones(n_arms) * 0.01
n_obs = np.zeros(n_arms)
sum_rewards = np.zeros(n_arms)

rewards_ts = []
regret_ts = []
arm_counts_ts = np.zeros(n_arms)

for t in range(n_rounds):
    # Sample from each arm's posterior
    theta_samples = []
    for i in range(n_arms):
        if n_obs[i] == 0:
            theta = np.random.normal(0, 0.1)
        else:
            posterior_mean = sum_rewards[i] / n_obs[i]
            posterior_std = np.sqrt(sigma_sq[i] / n_obs[i])
            theta = np.random.normal(posterior_mean, posterior_std)
        theta_samples.append(theta)
    
    # Pick arm with highest sample
    arm = np.argmax(theta_samples)
    
    reward = get_reward(arm)
    rewards_ts.append(reward)
    arm_counts_ts[arm] += 1
    
    # Update posterior (simplified: assume known variance)
    n_obs[arm] += 1
    sum_rewards[arm] += reward
    
    regret = best_mean - sectors[arm][1]
    regret_ts.append(regret)

cumulative_regret_ts = np.cumsum(regret_ts)
print(f"Thompson Sampling: Total regret = {cumulative_regret_ts[-1]:.4f}")

## Algorithm 4: Softmax (τ=0.1)

Select arms probabilistically based on their estimated values using Boltzmann distribution.

In [ ]:
# Softmax
np.random.seed(42)
tau = 0.1  # Temperature parameter

Q = np.zeros(n_arms)
N = np.zeros(n_arms)
rewards_sm = []
regret_sm = []
arm_counts_sm = np.zeros(n_arms)

for t in range(n_rounds):
    # Compute softmax probabilities
    exp_values = np.exp(Q / tau)
    probs = exp_values / exp_values.sum()
    
    # Sample arm according to probabilities
    arm = np.random.choice(n_arms, p=probs)
    
    reward = get_reward(arm)
    rewards_sm.append(reward)
    arm_counts_sm[arm] += 1
    
    N[arm] += 1
    Q[arm] += (reward - Q[arm]) / N[arm]
    
    regret = best_mean - sectors[arm][1]
    regret_sm.append(regret)

cumulative_regret_sm = np.cumsum(regret_sm)
print(f"Softmax (τ={tau}): Total regret = {cumulative_regret_sm[-1]:.4f}")

## Comparison Plot 1: Cumulative Regret

Lower regret = better performance. The algorithm that learns fastest has lowest cumulative regret.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

ax.plot(cumulative_regret_eg, label=f'ε-greedy (ε={epsilon})', linewidth=2, alpha=0.8)
ax.plot(cumulative_regret_ucb, label='UCB1', linewidth=2, alpha=0.8)
ax.plot(cumulative_regret_ts, label='Thompson Sampling', linewidth=2, alpha=0.8)
ax.plot(cumulative_regret_sm, label=f'Softmax (τ={tau})', linewidth=2, alpha=0.8)

ax.set_xlabel('Round', fontsize=12)
ax.set_ylabel('Cumulative Regret', fontsize=12)
ax.set_title('Algorithm Comparison: Cumulative Regret Over 1000 Rounds', fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n💡 Lower is better! Thompson Sampling typically has lowest regret.")

## Comparison Plot 2: Arm Selection Frequencies

How often did each algorithm select each arm? Ideal: mostly select the best arm (Metals).

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

algorithms = [
    (f'ε-greedy (ε={epsilon})', arm_counts_eg),
    ('UCB1', arm_counts_ucb),
    ('Thompson Sampling', arm_counts_ts),
    (f'Softmax (τ={tau})', arm_counts_sm)
]

sector_names = [s[0] for s in sectors]
colors = plt.cm.Set3(range(n_arms))

for idx, (name, counts) in enumerate(algorithms):
    ax = axes[idx]
    bars = ax.bar(sector_names, counts, color=colors, alpha=0.8)
    
    # Highlight best sector
    bars[1].set_edgecolor('red')
    bars[1].set_linewidth(3)
    
    ax.set_ylabel('Number of Pulls', fontsize=11)
    ax.set_title(name, fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='y')
    
    # Add percentage labels
    for i, (bar, count) in enumerate(zip(bars, counts)):
        pct = 100 * count / n_rounds
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10, 
                f'{pct:.1f}%', ha='center', va='bottom', fontsize=9)

plt.suptitle('Arm Selection Distribution (Red border = best sector)', fontsize=14, y=1.00)
plt.tight_layout()
plt.show()

print("\n💡 Best algorithms select 'Metals' (best sector) most often!")

## Summary Table: Algorithm Performance

Quantitative comparison across key metrics.

In [ ]:
best_arm_idx = 1  # Metals

results = {
    'Algorithm': [
        f'ε-greedy (ε={epsilon})',
        'UCB1',
        'Thompson Sampling',
        f'Softmax (τ={tau})'
    ],
    'Total Regret': [
        cumulative_regret_eg[-1],
        cumulative_regret_ucb[-1],
        cumulative_regret_ts[-1],
        cumulative_regret_sm[-1]
    ],
    '% Best Arm': [
        100 * arm_counts_eg[best_arm_idx] / n_rounds,
        100 * arm_counts_ucb[best_arm_idx] / n_rounds,
        100 * arm_counts_ts[best_arm_idx] / n_rounds,
        100 * arm_counts_sm[best_arm_idx] / n_rounds
    ],
    'Avg Reward': [
        np.mean(rewards_eg),
        np.mean(rewards_ucb),
        np.mean(rewards_ts),
        np.mean(rewards_sm)
    ]
}

df = pd.DataFrame(results)

print("\n" + "="*70)
print("ALGORITHM PERFORMANCE SUMMARY")
print("="*70)
print(df.to_string(index=False))
print("="*70)

best_algo = df.loc[df['Total Regret'].idxmin(), 'Algorithm']
print(f"\n🏆 Winner (lowest regret): {best_algo}")

print("\nKey insights:")
print("  • Thompson Sampling typically has lowest regret (no tuning needed!)")
print("  • UCB1 explores aggressively early, then converges")
print("  • ε-greedy is simple but wastes exploration on random arms")
print("  • Softmax is smooth but sensitive to temperature τ")

## Modify This

Experiment with different settings:

1. **Tune hyperparameters:**
   - Try `epsilon = 0.05` or `epsilon = 0.2`
   - Try `tau = 0.05` (more greedy) or `tau = 0.2` (more exploratory)

2. **Non-stationary rewards:**
   - At round 500, swap the means of 'Energy' and 'Metals'
   - Which algorithm adapts fastest?

3. **More arms:**
   - Add 5 more sectors with various mean/std combinations
   - Does the ranking change?

4. **Longer horizon:**
   - Try `n_rounds = 5000` - does Thompson Sampling pull further ahead?

5. **Different reward distributions:**
   - Use exponential or log-normal rewards instead of Gaussian

In [ ]:
# YOUR EXPERIMENTS HERE
# Copy the algorithm code above and modify parameters or reward distributions

## What's Next?

You just compared 4 major bandit algorithms!

**Continue learning:**
- `00_your_first_bandit.ipynb` - Thompson Sampling deep dive
- `01_ab_test_vs_bandit.ipynb` - Why bandits beat A/B tests
- `02_commodity_allocation_starter.ipynb` - Real commodity data
- `03_creator_bandit_playbook.ipynb` - Full content publishing playbook

**Deep dive:**
- **Module 1:** Regret bounds and theoretical guarantees for each algorithm
- **Module 2:** Thompson Sampling theory and Bayesian updating
- **Module 3:** When to use each algorithm (decision framework)
- **Module 6:** Non-stationary bandits (market regimes change!)
- **Module 7:** Exploration strategies and hyperparameter tuning

**Key takeaways:**
1. **Thompson Sampling:** Best overall, no tuning needed, Bayesian elegance
2. **UCB1:** Theoretical guarantees, aggressive early exploration
3. **ε-greedy:** Simple to implement, but wastes exploration
4. **Softmax:** Smooth exploration-exploitation, but sensitive to τ

**Rule of thumb:** Start with Thompson Sampling. It usually wins!

In [ ]:
key_takeaways(["Module 1:", "Module 2:", "Module 3:", "Module 6:", "Module 7:"])